#### 推荐的标准流程  
Raw→滤波→重采样→重参考→ICA→去除伪迹成分→Epochs 分段→后续分析
1. Raw 数据：原始 EEG/MEG 信号，包含所有通道的原始电压。
2. 滤波：对信号进行低通滤波，去除高频噪声。
3. 重采样：将信号重采样到 250 Hz，以匹配 MNE 默认采样率。
4. 重参考：将所有通道的参考电压设置为平均电压，以消除通道间差异。
5. ICA：独立成分分析，用于分离伪迹成分。
6. 去除伪迹成分：根据 ICA 分析结果，标记并去除伪迹成分。
7. Epochs 分段：将信号分段为事件相关时间窗口，用于后续分析。
8. 后续分析：使用 ERP、ERF 等指标进行分析。

In [ ]:
raw_proc = raw.copy()

# 1. 预处理
raw_proc.filter(1.0, 40.0)
raw_proc.resample(250)
raw_proc.set_eeg_reference("average")

# 2. 在连续 raw 上做 ICA
ica = ICA(n_components=20, method="fastica", random_state=42)
ica.fit(raw_proc, picks="eeg")

# 3. 标记要去除的 ICA 成分
ica.exclude = [0, 1]  # 举例：实际要根据图判断

# 4. 应用 ICA
raw_clean = raw_proc.copy()
ica.apply(raw_clean)

# 5. 再分段
epochs = mne.Epochs(
    raw_clean,
    events,
    event_id=event_id,
    tmin=-0.2,
    tmax=0.8,
    baseline=(None, 0),
    preload=True
)